# Food Delivery Time Prediction

## Objective
To predict whether a food order will be delivered fast or delayed using distance, traffic, weather and order details.

## Data Preprocessing
The data was checked for missing values and duplicates. The location columns were used to calculate the distance between the customer and restaurant.

## Feature Engineering
Haversine distance was calculated from customer and restaurant coordinates. A rush-hour feature was created from order time, and delivery time was converted into Fast or Delayed classes using the dataset median.


## K-Means Clustering
K-Means was used to group orders with similar delivery characteristics. The Elbow Method was used to choose the number of clusters.

## Hierarchical Clustering
Agglomerative clustering was used and the results were compared with K-Means.

## Prediction Models
Categorical variables such as weather, traffic, order priority, order time, and vehicle type were encoded using one-hot encoding. Logistic Regression and a simple Feedforward Neural Network were then used to predict whether an order would be Fast or Delayed.


## Note
The notebook mainly uses pandas, NumPy, Matplotlib and Seaborn. TensorFlow/Keras is used for the neural network when it is available.

## Delivery Time Analysis
Traffic, weather, distance, order priority and delivery-person experience were compared with delivery time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

print("Libraries loaded successfully.")

In [ ]:
file_path = "Food_Delivery_Time_Prediction.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

In [ ]:
print("Column Names:")
print(df.columns.tolist())

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

In [ ]:
display(df.describe(include="all"))

In [ ]:
numeric_columns = df.select_dtypes(
    include=[np.number]
).columns

categorical_columns = df.select_dtypes(
    exclude=[np.number]
).columns

for column in numeric_columns:
    df[column] = df[column].fillna(
        df[column].median()
    )

for column in categorical_columns:
    if df[column].isnull().any():
        df[column] = df[column].fillna(
            df[column].mode()[0]
        )

print("Missing values handled.")

print(
    "Total missing values:",
    df.isnull().sum().sum()
)

In [ ]:
def extract_coordinates(value):
    value = str(value).strip()

    value = value.replace("(", "")
    value = value.replace(")", "")

    parts = value.split(",")

    latitude = float(parts[0].strip())
    longitude = float(parts[1].strip())

    return latitude, longitude


customer_coordinates = df[
    "Customer_Location"
].apply(extract_coordinates)

restaurant_coordinates = df[
    "Restaurant_Location"
].apply(extract_coordinates)


df["Customer_Latitude"] = customer_coordinates.apply(
    lambda x: x[0]
)

df["Customer_Longitude"] = customer_coordinates.apply(
    lambda x: x[1]
)

df["Restaurant_Latitude"] = restaurant_coordinates.apply(
    lambda x: x[0]
)

df["Restaurant_Longitude"] = restaurant_coordinates.apply(
    lambda x: x[1]
)


display(
    df[
        [
            "Customer_Location",
            "Restaurant_Location",
            "Customer_Latitude",
            "Customer_Longitude",
            "Restaurant_Latitude",
            "Restaurant_Longitude"
        ]
    ].head()
)

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):

    earth_radius = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)

    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )

    return earth_radius * c


df["Haversine_Distance"] = df.apply(
    lambda row: haversine_distance(
        row["Customer_Latitude"],
        row["Customer_Longitude"],
        row["Restaurant_Latitude"],
        row["Restaurant_Longitude"]
    ),
    axis=1
)

print("Haversine distance created successfully.")

display(
    df[
        [
            "Distance",
            "Haversine_Distance",
            "Delivery_Time"
        ]
    ].head()
)

In [ ]:

df["Rush_Hour"] = (
    df["Order_Time"]
    .astype(str)
    .str.lower()
    .isin(["morning", "evening"])
    .astype(int)
)


median_delivery_time = df["Delivery_Time"].median()

df["Delivery_Status"] = np.where(
    df["Delivery_Time"] <= median_delivery_time,
    "Fast",
    "Delayed"
)

df["Target"] = np.where(
    df["Delivery_Status"] == "Fast",
    0,
    1
)

print("Rush Hour and target features created successfully.")

display(
    df[
        [
            "Order_Time",
            "Rush_Hour",
            "Delivery_Time",
            "Delivery_Status",
            "Target"
        ]
    ].head()
)

In [ ]:
cluster_features = [
    "Distance",
    "Haversine_Distance",
    "Delivery_Person_Experience",
    "Restaurant_Rating",
    "Customer_Rating",
    "Order_Cost",
    "Tip_Amount",
    "Rush_Hour"
]

cluster_data = df[cluster_features].copy()

cluster_data = cluster_data.astype(float)

print("Clustering data shape:", cluster_data.shape)

In [ ]:
cluster_features = [
    "Distance",
    "Haversine_Distance",
    "Delivery_Person_Experience",
    "Restaurant_Rating",
    "Customer_Rating",
    "Order_Cost",
    "Tip_Amount",
    "Rush_Hour"
]

cluster_data = df[cluster_features].copy()

cluster_data = cluster_data.apply(
    pd.to_numeric,
    errors="coerce"
)

cluster_data = cluster_data.fillna(
    cluster_data.median()
)

print("Clustering data prepared successfully.")
print("Shape:", cluster_data.shape)

display(cluster_data.head())

In [ ]:
cluster_mean = cluster_data.mean()
cluster_std = cluster_data.std()

cluster_std = cluster_std.replace(0, 1)

cluster_scaled = (cluster_data - cluster_mean) / cluster_std

print("Features standardized successfully.")
display(cluster_scaled.head())

In [ ]:
def manual_kmeans(data, k, max_iterations=100, random_state=42):
    rng = np.random.default_rng(random_state)
    data_array = data.to_numpy(dtype=float)

    initial_indices = rng.choice(
        len(data_array),
        size=k,
        replace=False
    )

    centroids = data_array[initial_indices].copy()
    previous_labels = None

    for iteration in range(max_iterations):
        distances = np.sqrt(
            ((data_array[:, np.newaxis, :] - centroids[np.newaxis, :, :]) ** 2).sum(axis=2)
        )
        labels = np.argmin(distances, axis=1)

        if previous_labels is not None and np.array_equal(labels, previous_labels):
            break

        previous_labels = labels.copy()
        new_centroids = []

        for cluster_number in range(k):
            cluster_points = data_array[labels == cluster_number]
            if len(cluster_points) > 0:
                new_centroids.append(cluster_points.mean(axis=0))
            else:
                new_centroids.append(centroids[cluster_number])

        centroids = np.array(new_centroids)

    final_distances = np.sqrt(
        ((data_array - centroids[labels]) ** 2).sum(axis=1)
    )
    inertia = np.sum(final_distances ** 2)

    return labels, centroids, inertia


In [ ]:
k_values = range(2, 9)
inertias = []

for k in k_values:
    labels, centroids, inertia = manual_kmeans(
        cluster_scaled,
        k=k,
        random_state=42
    )

    inertias.append(inertia)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

In [ ]:
best_k = 4

print("Selected number of clusters:", best_k)

In [ ]:
kmeans_labels, kmeans_centroids, kmeans_inertia = manual_kmeans(
    cluster_scaled,
    k=best_k,
    random_state=42
)

df["KMeans_Cluster"] = kmeans_labels

print("K-Means clustering completed successfully.")
print("Inertia:", round(kmeans_inertia, 2))

print("\nCluster distribution:")
print(df["KMeans_Cluster"].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(9, 6))

scatter = plt.scatter(
    df["Distance"],
    df["Delivery_Time"],
    c=df["KMeans_Cluster"],
    s=70
)

plt.xlabel("Distance")
plt.ylabel("Delivery Time")
plt.title("K-Means Clusters: Distance vs Delivery Time")
plt.colorbar(scatter, label="Cluster")
plt.grid(True)
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    df["Distance"],
    df["Delivery_Time"],
    df["Order_Cost"],
    c=df["KMeans_Cluster"],
    s=50
)

ax.set_xlabel("Distance")
ax.set_ylabel("Delivery Time")
ax.set_zlabel("Order Cost")
ax.set_title("K-Means Clusters - 3D View")
plt.show()


In [ ]:
cluster_summary = df.groupby("KMeans_Cluster").agg(
    Orders=("Delivery_Time", "count"),
    Average_Distance=("Distance", "mean"),
    Average_Delivery_Time=("Delivery_Time", "mean"),
    Average_Order_Cost=("Order_Cost", "mean"),
    Average_Tip=("Tip_Amount", "mean"),
    Average_Rating=("Restaurant_Rating", "mean")
).round(2)

display(cluster_summary)

In [ ]:
def manual_agglomerative(data, n_clusters):
    data_array = data.to_numpy(dtype=float)
    n_samples = len(data_array)

    clusters = [[i] for i in range(n_samples)]
    merge_history = []
    current_cluster_ids = list(range(n_samples))
    next_cluster_id = n_samples

    while len(clusters) > n_clusters:
        minimum_distance = float("inf")
        merge_a = None
        merge_b = None

        for i in range(len(clusters)):
            for j in range(i + 1, len(clusters)):
                points_a = data_array[clusters[i]]
                points_b = data_array[clusters[j]]
                centroid_a = points_a.mean(axis=0)
                centroid_b = points_b.mean(axis=0)
                distance = np.sqrt(np.sum((centroid_a - centroid_b) ** 2))

                if distance < minimum_distance:
                    minimum_distance = distance
                    merge_a = i
                    merge_b = j

        size_a = len(clusters[merge_a])
        size_b = len(clusters[merge_b])
        merge_history.append([
            current_cluster_ids[merge_a],
            current_cluster_ids[merge_b],
            minimum_distance,
            size_a + size_b
        ])

        clusters[merge_a].extend(clusters[merge_b])
        del clusters[merge_b]
        current_cluster_ids[merge_a] = next_cluster_id
        del current_cluster_ids[merge_b]
        next_cluster_id += 1

    labels = np.zeros(n_samples, dtype=int)
    for cluster_number, cluster_indices in enumerate(clusters):
        for index in cluster_indices:
            labels[index] = cluster_number

    return labels, clusters, np.array(merge_history, dtype=float)


In [ ]:
hierarchical_labels, hierarchical_clusters, merge_history = manual_agglomerative(
    cluster_scaled,
    n_clusters=best_k
)

df["Hierarchical_Cluster"] = hierarchical_labels

print("Hierarchical clustering completed successfully.")

print("\nCluster distribution:")
print(
    df["Hierarchical_Cluster"]
    .value_counts()
    .sort_index()
)


### Hierarchical Clustering Dendrogram
The dendrogram shows the sequence of cluster merges used by the agglomerative method.

In [ ]:
# Plot a simple dendrogram from the recorded merge history
plt.figure(figsize=(12, 6))

n_samples = len(df)
cluster_x = {i: i for i in range(n_samples)}
cluster_height = {i: 0.0 for i in range(n_samples)}
next_id = n_samples

for left_id, right_id, height, size in merge_history:
    left_x = cluster_x[int(left_id)]
    right_x = cluster_x[int(right_id)]
    left_h = cluster_height[int(left_id)]
    right_h = cluster_height[int(right_id)]

    plt.plot([left_x, left_x], [left_h, height], linewidth=0.8)
    plt.plot([right_x, right_x], [right_h, height], linewidth=0.8)
    plt.plot([left_x, right_x], [height, height], linewidth=0.8)

    cluster_x[next_id] = (left_x + right_x) / 2
    cluster_height[next_id] = height
    next_id += 1

plt.xlabel("Data Points")
plt.ylabel("Merge Distance")
plt.title("Hierarchical Clustering Dendrogram")
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
comparison = pd.DataFrame({
    "KMeans": df["KMeans_Cluster"],
    "Hierarchical": df["Hierarchical_Cluster"]
})

print("Cluster comparison:")
display(comparison.head(20))

In [ ]:
plt.figure(figsize=(9, 6))

scatter = plt.scatter(
    df["Distance"],
    df["Delivery_Time"],
    c=df["Hierarchical_Cluster"],
    s=70
)

plt.xlabel("Distance")
plt.ylabel("Delivery Time")
plt.title("Hierarchical Clustering: Distance vs Delivery Time")
plt.colorbar(scatter, label="Cluster")
plt.grid(True)
plt.show()

In [ ]:
hierarchical_summary = df.groupby("Hierarchical_Cluster").agg(
    Orders=("Delivery_Time", "count"),
    Average_Distance=("Distance", "mean"),
    Average_Delivery_Time=("Delivery_Time", "mean"),
    Average_Order_Cost=("Order_Cost", "mean"),
    Average_Tip=("Tip_Amount", "mean"),
    Average_Rating=("Restaurant_Rating", "mean")
).round(2)

display(hierarchical_summary)

In [ ]:
print("PHASE 2 - CLUSTERING SUMMARY")
print("=" * 50)

print("\nK-Means:")
print("Number of clusters:", best_k)
print("Inertia:", round(kmeans_inertia, 2))

print("\nK-Means Cluster Sizes:")
print(df["KMeans_Cluster"].value_counts().sort_index())

print("\nHierarchical Cluster Sizes:")
print(df["Hierarchical_Cluster"].value_counts().sort_index())

print("\nClustering analysis completed successfully.")

In [ ]:

df["Rush_Hour"] = (
    df["Order_Time"]
    .astype(str)
    .str.lower()
    .isin(["morning", "evening"])
    .astype(int)
)

median_delivery_time = df["Delivery_Time"].median()

df["Delivery_Status"] = np.where(
    df["Delivery_Time"] <= median_delivery_time,
    "Fast",
    "Delayed"
)

df["Target"] = np.where(
    df["Delivery_Status"] == "Fast",
    0,
    1
)

print("Rush Hour and delivery target created successfully.")
print("Median delivery time:", median_delivery_time)

display(
    df[
        [
            "Order_Time",
            "Rush_Hour",
            "Delivery_Time",
            "Delivery_Status",
            "Target"
        ]
    ].head()
)

In [ ]:
# Prepare numerical and categorical features for prediction
numerical_features = [
    "Distance",
    "Haversine_Distance",
    "Delivery_Person_Experience",
    "Restaurant_Rating",
    "Customer_Rating",
    "Order_Cost",
    "Tip_Amount",
    "Rush_Hour"
]
categorical_features = [
    "Weather_Conditions",
    "Traffic_Conditions",
    "Order_Priority",
    "Order_Time",
    "Vehicle_Type"
]

X_numeric = df[numerical_features].copy()
X_categorical = pd.get_dummies(
    df[categorical_features],
    drop_first=True,
    dtype=float
)

X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["Target"].copy()

X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median())

print("Classification data prepared successfully.")
print("Numerical features:", len(numerical_features))
print("Encoded features:", X.shape[1])
print("Target shape:", y.shape)
display(X.head())


In [ ]:
# Standardize using only the training data
X_mean = X.iloc[train_indices].mean()
X_std = X.iloc[train_indices].std().replace(0, 1)

X_train = ((X.iloc[train_indices] - X_mean) / X_std).to_numpy()
X_test = ((X.iloc[test_indices] - X_mean) / X_std).to_numpy()

y_train = y.iloc[train_indices].to_numpy()
y_test = y.iloc[test_indices].to_numpy()

print("Classification features standardized successfully.")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


In [ ]:
np.random.seed(42)

indices = np.arange(len(X))
np.random.shuffle(indices)

split_point = int(0.80 * len(indices))
train_indices = indices[:split_point]
test_indices = indices[split_point:]

print("Training samples:", len(train_indices))
print("Testing samples:", len(test_indices))


In [ ]:
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


def train_logistic_regression(X, y, learning_rate=0.01, epochs=3000):

    weights = np.zeros(X.shape[1])
    bias = 0.0

    for epoch in range(epochs):

        linear_output = np.dot(X, weights) + bias
        predictions = sigmoid(linear_output)

        error = predictions - y

        weight_gradient = np.dot(X.T, error) / len(X)
        bias_gradient = np.mean(error)

        weights -= learning_rate * weight_gradient
        bias -= learning_rate * bias_gradient

    return weights, bias

In [ ]:
logistic_weights, logistic_bias = train_logistic_regression(
    X_train,
    y_train,
    learning_rate=0.01,
    epochs=3000
)

print("Logistic Regression trained successfully.")

In [ ]:
logistic_probabilities = sigmoid(
    np.dot(X_test, logistic_weights) + logistic_bias
)

logistic_predictions = (logistic_probabilities >= 0.5).astype(int)

print("Logistic Regression predictions generated.")
print(logistic_predictions)

In [ ]:
def calculate_metrics(y_true, y_pred):

    true_positive = np.sum((y_true == 1) & (y_pred == 1))
    true_negative = np.sum((y_true == 0) & (y_pred == 0))
    false_positive = np.sum((y_true == 0) & (y_pred == 1))
    false_negative = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (
        (true_positive + true_negative)
        / len(y_true)
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if (true_positive + false_positive) > 0
        else 0
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if (true_positive + false_negative) > 0
        else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return accuracy, precision, recall, f1

In [ ]:
logistic_accuracy, logistic_precision, logistic_recall, logistic_f1 = calculate_metrics(
    y_test,
    logistic_predictions
)

print("LOGISTIC REGRESSION RESULTS")
print("=" * 40)
print("Accuracy :", round(logistic_accuracy, 4))
print("Precision:", round(logistic_precision, 4))
print("Recall   :", round(logistic_recall, 4))
print("F1 Score :", round(logistic_f1, 4))

The same encoded classification features are used for both Logistic Regression and the neural network.

## Feedforward Neural Network
A simple feedforward neural network is used to classify orders as Fast or Delayed. TensorFlow/Keras is used when available; a small NumPy fallback keeps the notebook runnable if TensorFlow is not installed.


In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    tensorflow_available = True
    print("TensorFlow/Keras is available.")
except Exception as error:
    tensorflow_available = False
    print("TensorFlow/Keras is not available in this environment.")
    print("The notebook will use the NumPy fallback so the remaining analysis can still run.")


In [ ]:
if tensorflow_available:
    tf.random.set_seed(42)

    neural_network = keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    neural_network.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history = neural_network.fit(
        X_train,
        y_train,
        epochs=50,
        batch_size=16,
        validation_split=0.2,
        verbose=0
    )

    print("Keras neural network trained successfully.")
else:
    print("Keras model was skipped because TensorFlow is not installed.")


In [ ]:
if tensorflow_available:
    nn_probabilities = neural_network.predict(X_test, verbose=0).ravel()
    nn_predictions = (nn_probabilities >= 0.5).astype(int)
    print("Keras predictions generated.")
else:
    # Small NumPy fallback for environments where TensorFlow is unavailable.
    class SimpleNeuralNetwork:
        def __init__(self, input_size, hidden_size=12, learning_rate=0.01):
            rng = np.random.default_rng(42)
            self.learning_rate = learning_rate
            self.W1 = rng.normal(0, 0.1, (input_size, hidden_size))
            self.b1 = np.zeros(hidden_size)
            self.W2 = rng.normal(0, 0.1, (hidden_size, 1))
            self.b2 = np.zeros(1)

        def relu(self, x):
            return np.maximum(0, x)

        def relu_derivative(self, x):
            return (x > 0).astype(float)

        def forward(self, X):
            self.Z1 = np.dot(X, self.W1) + self.b1
            self.A1 = self.relu(self.Z1)
            self.Z2 = np.dot(self.A1, self.W2) + self.b2
            self.A2 = sigmoid(self.Z2)
            return self.A2

        def train(self, X, y, epochs=2500):
            y = y.reshape(-1, 1)
            for _ in range(epochs):
                output = self.forward(X)
                dZ2 = output - y
                dW2 = np.dot(self.A1.T, dZ2) / len(X)
                db2 = np.mean(dZ2, axis=0)
                dA1 = np.dot(dZ2, self.W2.T)
                dZ1 = dA1 * self.relu_derivative(self.Z1)
                dW1 = np.dot(X.T, dZ1) / len(X)
                db1 = np.mean(dZ1, axis=0)
                self.W2 -= self.learning_rate * dW2
                self.b2 -= self.learning_rate * db2
                self.W1 -= self.learning_rate * dW1
                self.b1 -= self.learning_rate * db1

        def predict(self, X):
            probabilities = self.forward(X)
            return (probabilities.ravel() >= 0.5).astype(int)

    neural_network = SimpleNeuralNetwork(X_train.shape[1])
    neural_network.train(X_train, y_train)
    nn_predictions = neural_network.predict(X_test)
    print("NumPy fallback neural network trained successfully.")


In [ ]:
nn_accuracy, nn_precision, nn_recall, nn_f1 = calculate_metrics(
    y_test,
    nn_predictions
)

print("NEURAL NETWORK RESULTS")
print("=" * 40)
print("Accuracy :", round(nn_accuracy, 4))
print("Precision:", round(nn_precision, 4))
print("Recall   :", round(nn_recall, 4))
print("F1 Score :", round(nn_f1, 4))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Neural Network"
    ],
    "Accuracy": [
        logistic_accuracy,
        nn_accuracy
    ],
    "Precision": [
        logistic_precision,
        nn_precision
    ],
    "Recall": [
        logistic_recall,
        nn_recall
    ],
    "F1 Score": [
        logistic_f1,
        nn_f1
    ]
})

results = results.round(4)

display(results)

In [ ]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score"
]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(10, 6))

plt.bar(
    x - width / 2,
    results.iloc[0][metrics].values,
    width,
    label="Logistic Regression"
)

plt.bar(
    x + width / 2,
    results.iloc[1][metrics].values,
    width,
    label="Neural Network"
)

plt.xticks(x, metrics)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Logistic Regression vs Neural Network")
plt.legend()
plt.grid(axis="y")

plt.show()

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = np.zeros((2, 2), dtype=int)

    for actual, predicted in zip(y_true, y_pred):
        cm[actual, predicted] += 1

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Fast", "Delayed"],
        yticklabels=["Fast", "Delayed"]
    )

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.show()

    return cm


logistic_cm = plot_confusion_matrix(
    y_test,
    logistic_predictions,
    "Logistic Regression Confusion Matrix"
)

print(logistic_cm)

In [ ]:
nn_cm = plot_confusion_matrix(
    y_test,
    nn_predictions,
    "Neural Network Confusion Matrix"
)

print(nn_cm)

In [ ]:
traffic_summary = df.groupby("Traffic_Conditions")["Delivery_Time"].agg(
    ["count", "mean", "min", "max"]
).round(2)

print("Traffic Conditions vs Delivery Time")
display(traffic_summary)

In [ ]:
plt.figure(figsize=(8, 5))

traffic_means = df.groupby("Traffic_Conditions")["Delivery_Time"].mean()

plt.bar(
    traffic_means.index.astype(str),
    traffic_means.values
)

plt.xlabel("Traffic Conditions")
plt.ylabel("Average Delivery Time")
plt.title("Impact of Traffic Conditions on Delivery Time")
plt.xticks(rotation=20)
plt.grid(axis="y")

plt.show()

In [ ]:
weather_summary = df.groupby("Weather_Conditions")["Delivery_Time"].agg(
    ["count", "mean", "min", "max"]
).round(2)

print("Weather Conditions vs Delivery Time")
display(weather_summary)

In [ ]:
plt.figure(figsize=(8, 5))

weather_means = df.groupby("Weather_Conditions")["Delivery_Time"].mean()

plt.bar(
    weather_means.index.astype(str),
    weather_means.values
)

plt.xlabel("Weather Conditions")
plt.ylabel("Average Delivery Time")
plt.title("Impact of Weather Conditions on Delivery Time")
plt.xticks(rotation=20)
plt.grid(axis="y")

plt.show()

In [ ]:
priority_summary = df.groupby("Order_Priority")["Delivery_Time"].agg(
    ["count", "mean", "min", "max"]
).round(2)

print("Order Priority vs Delivery Time")
display(priority_summary)

In [ ]:
plt.figure(figsize=(8, 5))

priority_means = df.groupby("Order_Priority")["Delivery_Time"].mean()

plt.bar(
    priority_means.index.astype(str),
    priority_means.values
)

plt.xlabel("Order Priority")
plt.ylabel("Average Delivery Time")
plt.title("Order Priority vs Delivery Time")
plt.grid(axis="y")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["Distance"],
    df["Delivery_Time"],
    alpha=0.7
)

plt.xlabel("Distance")
plt.ylabel("Delivery Time")
plt.title("Distance vs Delivery Time")
plt.grid(True)

plt.show()

In [ ]:
status_counts = df["Delivery_Status"].value_counts()

print("Delivery Status Distribution:")
print(status_counts)

plt.figure(figsize=(6, 5))

plt.bar(
    status_counts.index,
    status_counts.values
)

plt.xlabel("Delivery Status")
plt.ylabel("Number of Orders")
plt.title("Fast vs Delayed Deliveries")
plt.grid(axis="y")

plt.show()

In [ ]:
cluster_delivery = df.groupby("KMeans_Cluster")["Delivery_Time"].mean().sort_values()

print("Average Delivery Time by K-Means Cluster")
display(cluster_delivery.round(2))

plt.figure(figsize=(8, 5))

plt.bar(
    cluster_delivery.index.astype(str),
    cluster_delivery.values
)

plt.xlabel("K-Means Cluster")
plt.ylabel("Average Delivery Time")
plt.title("Cluster Influence on Delivery Time")
plt.grid(axis="y")

plt.show()

In [ ]:
experience_summary = df.groupby(
    "Delivery_Person_Experience"
)["Delivery_Time"].mean().round(2)

display(experience_summary)

plt.figure(figsize=(8, 5))

plt.bar(
    experience_summary.index.astype(str),
    experience_summary.values
)

plt.xlabel("Delivery Person Experience")
plt.ylabel("Average Delivery Time")
plt.title("Delivery Experience vs Delivery Time")
plt.grid(axis="y")

plt.show()

In [ ]:
print("=" * 60)
print("FINAL MODEL COMPARISON")
print("=" * 60)

display(results)

print("\nClustering:")
print("K-Means clusters:", best_k)
print("K-Means inertia:", round(kmeans_inertia, 2))

print("\nDataset:")
print("Total orders:", len(df))
print("Fast orders:", int((df["Target"] == 0).sum()))
print("Delayed orders:", int((df["Target"] == 1).sum()))

In [ ]:
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

# Traffic
highest_traffic = traffic_means.idxmax()
highest_traffic_time = traffic_means.max()

print(
    f"1. The traffic condition with the highest average delivery time "
    f"is '{highest_traffic}' with approximately "
    f"{highest_traffic_time:.2f} minutes."
)

# Weather
highest_weather = weather_means.idxmax()
highest_weather_time = weather_means.max()

print(
    f"2. The weather condition with the highest average delivery time "
    f"is '{highest_weather}' with approximately "
    f"{highest_weather_time:.2f} minutes."
)

# Distance
distance_correlation = np.corrcoef(
    df["Distance"],
    df["Delivery_Time"]
)[0, 1]

print(
    f"3. The correlation between distance and delivery time "
    f"is approximately {distance_correlation:.2f}."
)

# Experience
fastest_experience = experience_summary.idxmin()

print(
    f"4. The delivery-person experience category with the lowest "
    f"average delivery time is '{fastest_experience}'."
)

# Cluster
fastest_cluster = cluster_delivery.idxmin()

print(
    f"5. K-Means Cluster {fastest_cluster} has the lowest "
    f"average delivery time among the identified clusters."
)

print("\nAnalysis completed successfully.")

In [ ]:
output_file = "Food_Delivery_Time_Prediction_Final.csv"

df.to_csv(
    output_file,
    index=False
)

print("Final dataset saved successfully.")
print("File:", output_file)

In [ ]:
print("=" * 70)
print("FOOD DELIVERY TIME PREDICTION - FINAL SUMMARY")
print("=" * 70)

print("\nPHASE 1 - DATA PREPROCESSING")
print("✓ Missing values handled")
print("✓ Location coordinates processed")
print("✓ Haversine distance calculated")
print("✓ Rush-hour feature created")
print("✓ Fast/Delayed target created")

print("\nPHASE 2 - CLUSTERING")
print("✓ K-Means clustering performed")
print("✓ Elbow method applied")
print("✓ Hierarchical clustering performed")
print("✓ Cluster delivery-time analysis completed")

print("\nPHASE 3 - PREDICTION")
print("✓ Logistic Regression implemented")
print("✓ Feedforward Neural Network implemented")
print("✓ Accuracy calculated")
print("✓ Precision calculated")
print("✓ Recall calculated")
print("✓ F1 Score calculated")
print("✓ Confusion matrices generated")

print("\nPHASE 4 - ANALYSIS")
print("✓ Traffic analysis completed")
print("✓ Weather analysis completed")
print("✓ Order priority analysis completed")
print("✓ Distance analysis completed")
print("✓ Delivery experience analysis completed")

print("\nPROJECT COMPLETED SUCCESSFULLY!")

## Conclusion
The analysis gives an overview of delivery patterns and the factors related to delivery time.